# Customer Churn Prediction - Preprocessing & Feature Engineering

This notebook handles data preprocessing and feature engineering for the churn prediction model.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## 1. Load Data

In [ ]:
# Load the cleaned data
df = pd.read_csv('../data/processed/customer_churn_cleaned.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Feature Engineering

In [ ]:
# Create additional features

# 1. Tenure groups (if not already created)
df['TenureGroup'] = pd.cut(df['TenureMonths'], bins=[0, 12, 24, 48, 72], 
                            labels=['New', 'Regular', 'Established', 'Loyal'])

# 2. Average monthly spend
df['AvgMonthlySpend'] = df['TotalCharges'] / (df['TenureMonths'] + 1)

# 3. Service utilization rate
total_services = 6  # OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies
df['ServiceUtilization'] = df['ServiceCount'] / total_services

# 4. Is high value customer
df['HighValueCustomer'] = (df['MonthlyCharges'] > df['MonthlyCharges'].median()).astype(int)

# 5. Price per service
df['PricePerService'] = df['MonthlyCharges'] / (df['ServiceCount'] + 1)

# 6. Has protection bundle
df['HasProtectionBundle'] = ((df['OnlineSecurity'] == 1) & 
                             (df['OnlineBackup'] == 1) & 
                             (df['DeviceProtection'] == 1)).astype(int)

# 7. Tenure-to-charge ratio
df['TenureToChargeRatio'] = df['TenureMonths'] / (df['MonthlyCharges'] + 1)

print("✓ Feature engineering completed")
print(f"  New features created: 7")
print(f"  Total features: {df.shape[1]}")

## 3. Encode Categorical Variables

In [ ]:
# Identify categorical columns
categorical_cols = ['Gender', 'State', 'ContractType', 'InternetService', 'PaymentMethod', 'TenureGroup']

# Create a copy for encoding
df_encoded = df.copy()

# One-hot encode categorical variables
df_encoded = pd.get_dummies(df_encoded, columns=categorical_cols, drop_first=True)

print(f"✓ Categorical encoding completed")
print(f"  Shape after encoding: {df_encoded.shape}")

## 4. Prepare Features and Target

In [ ]:
# Drop non-predictive columns
columns_to_drop = ['CustomerID', 'ChargeGroup']
df_encoded = df_encoded.drop(columns=[col for col in columns_to_drop if col in df_encoded.columns])

# Separate features and target
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

print(f"Feature matrix shape: {X.shape}")
print(f"Target variable shape: {y.shape}")
print(f"\nChurn distribution:")
print(y.value_counts(normalize=True))

## 5. Train-Test Split

In [ ]:
# Split data (80-20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"\nTraining churn distribution:")
print(y_train.value_counts(normalize=True))
print(f"\nTest churn distribution:")
print(y_test.value_counts(normalize=True))

## 6. Feature Scaling

In [ ]:
# Standardize numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame to preserve column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("✓ Feature scaling completed")

## 7. Save Processed Data

In [ ]:
# Save the preprocessed data
X_train_scaled.to_csv('../data/processed/X_train.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save the scaler
import pickle
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("✓ All preprocessed data and scaler saved successfully")
print("\nReady for modeling phase!")